# LLM Fundamentals: Building Large Language Models from the Ground Up

### Google Colab Practice Notebook

Run the cells from top to bottom. Select a GPU runtime before starting.


# Preface: How to Read This Book

## Goals of This Book

This book is a hands-on guide designed to teach you **how Large Language Models (LLMs)**, such as ChatGPT and GPT-4, **work internally** by implementing them from scratch through code.

If you only study theory, you might understand that "attention is important," but you won't grasp *why* or *how* it actually works in practice. Conversely, if you only follow code, you will end up copy-pasting without understanding the underlying principles. This book bridges that gap by combining both:

- **Theory**: The beginning of each chapter explains concepts using analogies so they can be understood even without complex diagrams and formulas.
- **Practice**: Every chapter includes Python code that is ready to run in Google Colab. Using PyTorch, we will build tokenizers, embeddings, attention mechanisms, and Transformer blocks line by line, ultimately training a small GPT model to generate text.

By following this book to the end, you will implement the following:

1. A BPE tokenizer that splits text into tokens
2. Embedding layers that convert words into vectors
3. Self-Attention and Multi-Head Attention
4. Transformer blocks (LayerNorm, FeedForward, Residual Connections)
5. The complete architecture of a mini GPT model
6. Training loops and loss functions
7. Text generation sampling strategies (top-k, top-p, temperature)
8. Loading pre-trained models from Hugging Face
9. Lightweight fine-tuning using LoRA

## Prerequisites

It is perfectly fine if you only know basic Python syntax or are new to deep learning. However, the journey will be much smoother if you have heard of the following concepts:

- Matrix multiplication (dot products, dimensions)
- Basic neural network structure (input, hidden, and output layers)
- Gradient descent

If you are unfamiliar with these, we will provide brief explanations within each chapter as needed.

## Using GitHub Repositories and Google Colab

All practice code and demo scripts for this book are maintained in the following GitHub repository. You can download the notebook files directly or open them in your browser.

- **GitHub Repository**: [https://github.com/dirmich/mini_gpt_release](https://github.com/dirmich/mini_gpt_release)

Google Colab is a Python execution environment that allows you to use GPUs for free in your browser. Start without any local installation by following these steps:

1. Go to [colab.research.google.com](https://colab.research.google.com) and log in.
2. Use `File > New Notebook` or `File > Upload notebook` to load the `.ipynb` files downloaded from the GitHub repository.
3. In `Runtime > Change runtime type`, set the hardware accelerator to **GPU** (T4 or higher recommended).
4. Copy each code block into a cell and run it using `Shift + Enter`.

Whenever a code block appears in the book, it will be marked with a comment like this:


In [ ]:
# ── Colab Cells ──


When you see this marker, treat it as a new cell to paste and execute. Since subsequent cells within the same chapter rely on variables and functions defined in previous cells, **you must run them in order**.

## Module Structure Guide

Instead of one massive script, the practice code is organized into modules based on their roles, just like real open-source LLM libraries.

```
mini_gpt/
├── tokenizer.py       # Chapter 2: BPE Tokenizer
├── embedding.py        # Chapter 3: Token Embedding + Positional Encoding
├── attention.py         # Chapter 4: Self-Attention, Multi-Head Attention
├── transformer_block.py # Chapter 5: Transformer Block
├── model.py             # Chapter 6: Assembling the Full GPT Model
├── train.py              # Chapter 7: Training Loop
└── generate.py           # Chapter 8: Text Generation
```

In Colab, rather than creating multiple files manually, we use the `%%writefile` magic command to save each module as a file and then `import` them in subsequent cells. This allows you to experience a modularized structure—just like a real-world project—within the Colab environment.

Now, let's begin.


# What is an LLM?

## Definition of a Language Model

Simply put, a Language Model (LM) is **"a machine that predicts the next word based on probability."**

> "For dinner tonight, I ate ___."

Words like "rice," "chicken," or "ramen" have a high probability for the blank, while words like "car" or "spaceship" have a low probability. By learning from vast amounts of text, a language model statistically learns these probability distributions. The process by which models like ChatGPT generate sentences is fundamentally an iterative task of "predicting the next token and appending it."

The term "Large" is applied for two main reasons:

1. **Massive Parameter Count**: They possess hundreds of millions to trillions of trainable weights.
2. **Massive Training Data**: They learn from hundreds of billions to trillions of tokens, including internet text, books, and code.

## Why Transformer?

The **Transformer** architecture, proposed in the 2017 paper "Attention Is All You Need," is the foundation for almost all modern LLMs. Before Transformers, Recurrent Neural Networks (RNNs) were primarily used. However, RNNs process sentences one word at a time in sequence, making them slow and causing them to forget information from the beginning of long sentences.

Transformers use an **Attention** mechanism to look at all words in a sentence simultaneously and calculate how much each word relates to every other word. This allows for:

- **Parallel Processing**: Enabling faster training.
- **Long-range Dependencies**: Capturing relationships between distant words even in long sentences.

The mini GPT we will build in this book is a scaled-down version of this exact Transformer architecture.

## At a Glance: How GPT Models Work

GPT (Generative Pre-trained Transformer) models operate through the following pipeline.

```
Text Input
   │
   ▼
[Tokenizer] Text → Token ID Sequence
   │
   ▼
[Embedding] Token ID → Vector, Add Positional Information
   │
   ▼
[Transformer Block × N] Repeat Self-Attention + FeedForward
   │
   ▼
[Output Layer] Calculate Probability Distribution for the Next Token
   │
   ▼
[Sampling] Select Next Token from Probability Distribution
   │
   ▼
Append Generated Token to Input and Repeat
```

Chapters 2 through 8 of this book correspond exactly to each stage of this pipeline. In other words, this diagram is the roadmap for this entire book.

## Pre-training and Fine-tuning

LLMs are typically created in two stages:

- **Pre-training**: The model repeatedly learns "next-token prediction" using massive amounts of text. Once this stage is complete, the model possesses grammar, common sense, and world knowledge to some extent. However, it does not yet know "how to answer questions."
- **Fine-tuning**: The pre-trained model undergoes additional training with a small amount of data to suit specific purposes (conversation, instruction following, code generation, etc.). This includes Supervised Fine-Tuning (SFT), Reinforcement Learning from Human Feedback (RLHF), and lightweight techniques like LoRA.

This book covers pre-training our mini GPT from scratch in Chapters 6–7, and fine-tuning a real pre-trained model using LoRA in Chapter 11.

## Preparing the Development Environment (First Colab Cell)

Now, let's run our first code. We will check if the GPU is properly detected in Colab and install the necessary packages.


In [ ]:
# ── Colab Cell ──
# Check GPU
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device to use:", device)


In [ ]:
# ── Colab Cell ──
# Install packages used throughout this book
!pip install -q torch transformers datasets tiktoken regex peft accelerate


If `torch.cuda.is_available()` returns `False`, please check if you have set the runtime type to T4 GPU under `Runtime > Change runtime type`. While you can proceed with the exercises in this book using only a CPU, the training stage in Chapter 7 will be much faster with a GPU.

Starting from the next chapter, we will build a **Tokenizer**—the component that converts text into numbers that the model can understand.


# Tokenization: Turning Text into Numbers

## Why Tokens are Necessary

Neural networks only understand numbers. You cannot feed a string like "Hello" directly into a model; it must be converted into a sequence of integer IDs. This conversion process is called **Tokenization**, and the tool containing the transformation rules is called a **Tokenizer**.

The simplest method is "splitting by word and assigning a number to each word." However, this approach faces two problems:

1. It cannot handle out-of-vocabulary (OOV) words (neologisms, typos, foreign languages).
2. If the number of unique words becomes too large, the vocabulary size grows excessively.

Conversely, "splitting by character" results in a small vocabulary, but sentences become excessively long, and individual characters carry almost no meaning, making training inefficient.

## BPE: A Compromise Between Both Approaches

**BPE (Byte Pair Encoding)** is the method used by most actual GPT-style models. It builds a vocabulary by iteratively merging frequently occurring pairs of characters. For example, if words like "low", "lower", and "lowest" appear frequently, it merges pieces that often appear together into larger units, such as `l`+`o` $\rightarrow$ `lo`, `lo`+`w` $\rightarrow$ `low`.

As a result, frequently used words become single tokens, while rare words are split into multiple subword tokens. This allows us to control the vocabulary size while still being able to represent unseen words through combinations of subwords.

## BPE Training Algorithm

1. Split the text into individual characters.
2. Find the most frequent pair of adjacent tokens.
3. Merge that pair into a single new token.
4. Repeat steps 2–3 until the desired vocabulary size is reached.

Now, let's implement this ourselves. In Colab, enter the following code in order into new cells.


In [ ]:
# ── Colab Cell ──
# Create mini_gpt package folder
import os
os.makedirs("mini_gpt", exist_ok=True)


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/tokenizer.py
"""BPE (Byte Pair Encoding) Tokenizer - Implementation from scratch"""
from collections import Counter, defaultdict


class BPETokenizer:
    def __init__(self):
        self.merges = {}          # (token1, token2) -> merged new token ID
        self.vocab = {}           # Token ID -> byte sequence
        self.token_to_id = {}     # String -> Token ID

    def _get_pair_counts(self, token_ids_list):
"""Counts the frequency of adjacent token pairs in all sequences."""
        counts = Counter()
        for ids in token_ids_list:
            for a, b in zip(ids, ids[1:]):
                counts[(a, b)] += 1
        return counts

    def _merge(self, ids, pair, new_id):
"""Merges all pairs within ids into new_id."""
        merged = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
                merged.append(new_id)
                i += 2
            else:
                merged.append(ids[i])
                i += 1
        return merged

    def train(self, text, vocab_size=500):
"""Trains a BPE vocabulary from text."""
# Step 1: Initial vocabulary consists of 256 UTF-8 bytes
        for i in range(256):
            self.vocab[i] = bytes([i])

# Convert text to a list of byte-level integers
        ids = list(text.encode("utf-8"))
token_ids_list = [ids]  # Only one item in the list as there is only one document

        num_merges = vocab_size - 256
        for step in range(num_merges):
            pair_counts = self._get_pair_counts(token_ids_list)
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = 256 + step
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_id
            token_ids_list = [self._merge(ids, best_pair, new_id) for ids in token_ids_list]

        return self

    def encode(self, text):
"""Converts text into a list of token IDs."""
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            pair_counts = self._get_pair_counts([ids])
            # Among merges created during training, apply the earliest-created pair first
            candidates = [p for p in pair_counts if p in self.merges]
            if not candidates:
                break
            best_pair = min(candidates, key=lambda p: self.merges[p])
            ids = self._merge(ids, best_pair, self.merges[best_pair])
        return ids

    def decode(self, ids):
"""Restores a list of token IDs back to text."""
        byte_seq = b"".join(self.vocab[i] for i in ids)
        return byte_seq.decode("utf-8", errors="replace")


Now, let's train this tokenizer and perform actual encoding/decoding.


In [ ]:
# ── Colab Cell ──
from mini_gpt.tokenizer import BPETokenizer

sample_text = """
Artificial Intelligence is a subfield of computer science that aims to artificially implement human learning, reasoning, and perception capabilities.
Large Language Models (LLMs) learn language by training on massive text datasets to predict the next word.
The Transformer architecture uses the attention mechanism as its core component.
""" * 20  # Repeat to make merge patterns more visible

tokenizer = BPETokenizer()
tokenizer.train(sample_text, vocab_size=500)

test = "Transformers use attention."
encoded = tokenizer.encode(test)
decoded = tokenizer.decode(encoded)

print("Original:", test)
print("Number of token IDs:", len(encoded))
print("Token IDs:", encoded)
print("Decoded text:", decoded)
print("Is it identical to the original?", test == decoded)


A major advantage of this method is that it starts from the byte level, allowing it to process any text—including Korean, English, emojis, and special characters. If you run it repeatedly, you can observe in `tokenizer.vocab` how frequently occurring Korean fragments (e.g., "language model", "transformer") are merged into single tokens.

## In Practice, We Use `tiktoken`

The tokenizer we built is for educational purposes to understand the underlying principles. In practice, we use libraries that are much faster and have-already-built vocabularies from massive datasets. OpenAI's `tiktoken` is a prime example.


In [ ]:
# ── Colab Cell ──
import tiktoken

enc = tiktoken.get_encoding("gpt2")
test = "Transformers use attention mechanisms."
ids = enc.encode(test)
print("Token IDs:", ids)
print("Number of tokens:", len(ids))
print("Decoded:", enc.decode(ids))

# Check pieces cut by token units
for tid in ids:
    print(tid, "->", repr(enc.decode([tid])))


When we train our mini-GPT in Chapters 6 and 7, we will use the `BPETokenizer` we built to maintain the principles, but keep in mind that real-world services use proven libraries like `tiktoken`.

In the next chapter, we will build an embedding layer that converts these token IDs into **vectors** so that the model can process them with semantic meaning.


# Embedding and Positional Encoding

## Why Token IDs Alone Are Not Enough

After passing through a tokenizer, text becomes a list of integers like `[1023, 45, 892, ...]`. However, these integers carry no inherent meaning. Just because token ID 1023 and 1024 are adjacent numbers does not mean the tokens they represent have similar meanings.

Therefore, we need a process to convert each token ID into a **real-valued vector that carries semantic meaning**. This is called **Embedding**. Through training, embedding vectors are adjusted so that "tokens with similar meanings are placed closer together in the vector space." For example, if training is successful, the embedding vectors for "cat" and "dog" will be closer to each other than those for "cat" and "car."

From an implementation perspective, an embedding layer is essentially a simple **lookup table**. If the vocabulary size is $V$ and the embedding dimension is $d$, we create a matrix of size $(V, d)$ and use the token ID as the row index to retrieve the corresponding vector.

## Why Positional Information Is Needed Separately

The Transformer's attention mechanism processes all tokens in a sentence in parallel. While this provides a massive advantage in terms of speed, it has a side effect: **it does not automatically provide information about "which token is at which position."** RNNs naturally reflect positional information because they process sequences step-by-step, but Transformers cannot distinguish between "I ate rice" and "Rice I ate" without order information.

To solve this, we use **Positional Encoding**, which adds a unique vector to each position (1st, 2nd, 3rd...). There are several approaches:

- **Sine/Cosine Function-based** (Original Transformer paper): Uses sine and cosine values of different frequencies for each position. It has no trainable parameters.
- **Learned Positional Embeddings** (GPT-2, etc.): Treats positional indices as a separate embedding table to be learned, just like tokens.

In this book, we use the learned positional embedding method, which is identical to GPT-2. This approach is simpler to implement and easier to understand.

## Implementation


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/embedding.py
"""Token Embedding + Positional Embedding"""
import torch
import torch.nn as nn


class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # Token ID -> Vector (Lookup Table), Size: (vocab_size, embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Position Index -> Vector, Size: (max_seq_len, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids):
        # token_ids: (batch_size, seq_len)
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        # positions: (1, seq_len) -> Applied to the entire batch via broadcasting

        token_vecs = self.token_embedding(token_ids)      # (batch, seq_len, embed_dim)
        position_vecs = self.position_embedding(positions)  # (1, seq_len, embed_dim)

        # Adding these two vectors carries information like "this token at this specific position"
        embeddings = token_vecs + position_vecs
        return self.dropout(embeddings)


## Verifying Operation Manually


In [ ]:
# ── Colab Cell ──
from mini_gpt.embedding import GPTEmbedding
import torch

vocab_size = 500      # Vocabulary size of the tokenizer created in Chapter 2
embed_dim = 64         # Vector dimension to represent each token
max_seq_len = 32        # Maximum number of tokens the model can see at once

embedding_layer = GPTEmbedding(vocab_size, embed_dim, max_seq_len)

# Fake token ID batch: (batch_size=2, seq_len=5)
fake_token_ids = torch.tensor([
    [10, 25, 3, 88, 42],
    [7,  7,  7, 7,  7],
])

output = embedding_layer(fake_token_ids)
print("Input shape:", fake_token_ids.shape)
print("Output shape:", output.shape)   # Should be (2, 5, 64)
print("Partial embedding vector for the first token of the first sentence:", output[0, 0, :8])


The second sentence of `fake_token_ids` repeats the same token `7` five times. If there were no positional embeddings, the output vectors for these five positions would be identical. Check this yourself with the code below.


In [ ]:
# ── Colab Cell ──
# Check if embeddings differ even for the same token when positions are different
second_sentence_output = output[1]  # (5, 64)
for i in range(5):
    print(f"Partial vector at position {i}:", second_sentence_output[i, :4].tolist())


All five vectors are output differently. This is exactly how positional encoding injects "order information" into the embeddings.

## Criteria for Determining Embedding Dimension (`embed_dim`)

`embed_dim` is a hyperparameter that determines how many real numbers represent a single token. A larger value allows for richer semantic representation, but it also increases computational load and memory usage. For reference, here are the approximate scales of actual models:

| Model | Embedding Dimension | Number of Layers |
|---|---|---|
| GPT-2 Small | 768 | 12 |
| GPT-2 XL | 1600 | 48 |
| Mini GPT in this book | 64~128 | 4~6 |

This book is designed with a very small scale so that training finishes within minutes even on a free Colab GPU. The principles are identical to large-scale models; it scales simply by increasing the dimensions and number of layers.

In the next chapter, we will implement **Attention**, the core mechanism through which these embedding vectors exchange information.


# Attention Mechanism

## Core Idea: Meaning Changes with Context

In the English sentences "I sat by the river bank," "I deposited money at the bank," and "the plane banked to the left," the word "bank" carries entirely different meanings: a riverside, a financial institution, and a tilting motion. Embedding alone cannot express this difference if it assigns the same vector to the token "bank" every time.

**Attention** is a mechanism that calculates weights to determine "how much each word should refer to other words in the sentence," re-combining vectors to fit the context. The model learns to focus (attend) on words such as "river," "money," or "plane" around "bank" and infer the meaning that fits the sentence.

## Query, Key, and Value Analogy

Attention can be compared to the process of searching for a book in a library.

- **Query**: A question vector representing "This is the information I need right now." It is created by the word currently being processed.
- **Key**: An index vector where each word says, "This is the type of information I possess."
- **Value**: The actual content vector that contains the information to be passed on.

We compare the Query with the Keys of all words (via dot product) to calculate similarity scores, normalize these scores into probabilities using `softmax`, and then compute a weighted sum of the Values based on those probabilities. In short, Attention is simply "creating a new vector by reflecting more heavily on the content of words that have high relevance to the query."

Mathematically, it is expressed as follows:

```
Attention(Q, K, V) = softmax( Q · K^T / sqrt(d_k) ) · V
```

- `Q · K^T`: Calculates the similarity score matrix via dot product of Query and Key.
- Dividing by `sqrt(d_k)`: Prevents the dot product values from growing too large as dimensionality increases, which would cause the softmax to saturate (scaling).
- `softmax`: Converts scores into a probability distribution that sums to 1.
- Multiplying by `V` at the end: Calculates the new vector as a weighted average of the Values.

## Self-Attention Implementation


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/attention.py
"""Implementation of Self-Attention and Multi-Head Attention"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelfAttention(nn.Module):
    """Most basic single-head Self-Attention"""

    def __init__(self, embed_dim, head_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # Linear transformation to create Q, K, V. Input and output dimensions may differ (head_dim)
        self.query_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.key_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.value_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

        # Causal mask (lower triangular matrix) to prevent looking at future tokens
        # Since GPT's goal is "next word prediction," referring to future information would be cheating.
        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        batch_size, seq_len, _ = x.shape

        Q = self.query_proj(x)   # (batch, seq_len, head_dim)
        K = self.key_proj(x)     # (batch, seq_len, head_dim)
        V = self.value_proj(x)   # (batch, seq_len, head_dim)

        head_dim = Q.shape[-1]
        # Similarity scores: (batch, seq_len, seq_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(head_dim)

        # Masking future positions with -inf -> becomes 0 probability after softmax
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)  # Sum of each row is 1
        attn_weights = self.dropout(attn_weights)

        output = attn_weights @ V  # (batch, seq_len, head_dim)
        return output, attn_weights


class MultiHeadAttention(nn.Module):
    """Multiple heads calculate attention in parallel from different perspectives"""

    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        # Calculate Q, K, V all at once (processed with a single linear layer for efficiency)
        qkv = self.qkv_proj(x)  # (batch, seq_len, embed_dim*3)
        Q, K, V = qkv.chunk(3, dim=-1)

        # (batch, seq_len, embed_dim) -> (batch, num_heads, seq_len, head_dim)
        def split_heads(t):
            t = t.view(batch_size, seq_len, self.num_heads, self.head_dim)
            return t.transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V  # (batch, num_heads, seq_len, head_dim)

        # Concatenate back to (batch, seq_len, embed_dim)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        return self.out_proj(out)


## Verification and Attention Weight Visualization


In [ ]:
# ── Colab Cell ──
from mini_gpt.attention import SelfAttention, MultiHeadAttention
import torch

embed_dim = 32
seq_len = 6
batch_size = 1

x = torch.randn(batch_size, seq_len, embed_dim)

attn = SelfAttention(embed_dim, head_dim=embed_dim, max_seq_len=16)
output, weights = attn(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)  # (1, 6, 6)
print("\nAttention weight matrix (Row=current token, Column=referenced token):")
for row in weights[0]:
    print([round(v, 2) for v in row.tolist()])


Looking at the output matrix, you can see that the upper triangular part (future positions) consists entirely of zeros. This is the result of `causal_mask` blocking future information. Each row sums exactly to 1 because softmax creates a probability distribution.


In [ ]:
# ── Colab Cell ──
# Check Multi-Head Attention in the same way
mha = MultiHeadAttention(embed_dim=32, num_heads=4, max_seq_len=16)
out = mha(x)
print("Multi-Head Attention output shape:", out.shape)  # (1, 6, 32) - Maintains the same shape as input


## Why Use Multiple Heads?

A single attention head calculates relationships between words from only one perspective. For example, one head might focus on grammatical relationships (subject-verb), while another focuses on semantic similarity. **Multi-Head Attention** divides the embedding dimension into several parts (heads), allows each part to calculate attention independently, and then concatenates them. This enables the model to capture multiple types of relationships in parallel.

In the next chapter, we will implement the **Transformer Block** (LayerNorm, FeedForward, Residual Connection) that wraps this attention mechanism to ensure stable training.


# Assembling Transformer Blocks

The Multi-Head Attention we built in the previous chapter does not learn well on its own. A real Transformer uses three additional mechanisms around the attention mechanism: **Residual Connections**, **Layer Normalization**, and **Feed-Forward Networks**. Combining these three with attention into a single unit is called a **Transformer Block**, and models are constructed by stacking these blocks multiple layers deep.

## Residual Connection: Preventing Information Loss

As layers are stacked deeper, original information can fade through repeated transformations, or gradients may vanish during training. Residual connections solve this with a simple trick.

```
output = attention(x) + x
```

In other words, instead of using the transformed result directly, **we add the input $x$ back to it.** This ensures that even in the worst-case scenario (even if attention provides no help), at least the input information is passed to the next layer. It is like creating an additional "shortcut." This was first proposed in ResNet for image models and is used identically in Transformers.

## Layer Normalization: Stabilizing Training

As neural networks grow deeper, the scale of values can fluctuate wildly as they pass through each layer, making training unstable. LayerNorm normalizes the vector of each token to have a mean of 0 and a variance of 1, then multiplies and adds learnable scale and shift values.

```
LayerNorm(x) = γ · (x - mean) / sqrt(variance + ε) + β
```

Here, $\gamma$ (gamma) and $\beta$ (beta) are trainable parameters that allow the model to re-scale the output as needed after normalization.

## Feed-Forward Network: Token-wise Non-linear Transformation

While attention is responsible for calculating "relationships between tokens," the feed-forward network processes each token's vector independently and more deeply. The structure is simple: it places a non-linear activation function (GELU) between two linear layers—one that expands the dimension (usually by 4x) and another that reduces it back to the original dimension.

```
FFN(x) = Linear2( GELU( Linear1(x) ) )
```

The reason for expanding and then shrinking the dimensions is to provide enough "room" in a higher-dimensional space to combine various features.

## Pre-LN Architecture

Depending on where LayerNorm is placed, there are two approaches: Post-LN and Pre-LN. The original Transformer paper used Post-LN (normalization after Attention/FFN), but most models since GPT-2 have adopted **Pre-LN** (normalization before Attention/FFN). This is because Pre-LN is much more stable during the early stages of training. This book also uses Pre-LN.

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

## Implementation


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/transformer_block.py
"""Transformer Block: Attention + FeedForward + LayerNorm + Residual"""
import torch.nn as nn
from mini_gpt.attention import MultiHeadAttention


class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.feed_forward = FeedForward(embed_dim, hidden_mult=4, dropout=dropout)

    def forward(self, x):
        # Pre-LN structure: Normalization -> Sub-layer -> Residual Connection
        x = x + self.attention(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x


## Verifying Operation


In [ ]:
# ── Colab Cell ──
from mini_gpt.transformer_block import TransformerBlock
import torch

embed_dim = 64
seq_len = 10
batch_size = 2

x = torch.randn(batch_size, seq_len, embed_dim)
block = TransformerBlock(embed_dim=embed_dim, num_heads=4, max_seq_len=32)

output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)  # Must be identical to input - the reason why blocks can be stacked multiple times
print("Number of parameters:", sum(p.numel() for p in block.parameters()))


It is crucial that the input and output shapes of the block are identical. This allows us to stack these blocks like LEGO bricks as many times as we want.


In [ ]:
# ── Colab Cell ──
# Briefly check if residual connections actually stabilize gradients
import torch.nn as nn

# Check if gradients do not vanish even when stacking 12 blocks deeply
blocks = nn.ModuleList([
    TransformerBlock(embed_dim=64, num_heads=4, max_seq_len=32) for _ in range(12)
])

x = torch.randn(1, 10, 64, requires_grad=True)
out = x
for b in blocks:
    out = b(out)

loss = out.sum()
loss.backward()

print("Mean absolute value of input x gradient:", x.grad.abs().mean().item())
print("-> Not being close to 0 means the gradient is well-propagated even through 12 layers.")


In the next chapter, we will stack $N$ of these Transformer blocks and connect them with embedding and output layers to assemble a complete mini-GPT model.


# Assembling the Mini GPT Model

Here is a summary of the components we have built so far:

- `GPTEmbedding`: Token ID $\rightarrow$ Vector (Token Embedding + Positional Embedding)
- `TransformerBlock`: Attention + Feed-forward (We will stack $N$ of these)
- The final requirement: An output layer that converts the outputs of the transformer blocks back into a **"probability distribution over the vocabulary size."**

In this chapter, we will assemble these components to create a complete GPT model class.

## Output Layer (LM Head)

After passing through the transformer blocks, the vector at each token position still has an `embed_dim` size. To convert this vector into "the probability that the next token is each word in the vocabulary," we need a linear layer that transforms `embed_dim` $\rightarrow$ `vocab_size`. This is called the **LM Head (Language Modeling Head)**.

```
logits = Linear(embed_dim -> vocab_size)(Transformer output)
```

Applying softmax to the logits results in a probability distribution over the entire vocabulary. In actual training, we do not apply softmax explicitly; instead, we let `CrossEntropyLoss` handle both the softmax and the logarithm internally (this is more numerically stable).

## Weight Tying

Many models, including GPT-2, **share** weights between the token embedding matrix and the LM Head. This is because they perform similar roles in "token $\leftrightarrow$ vector" conversion (one maps ID $\rightarrow$ vector, the other maps vector $\rightarrow$ scores per ID). This technique significantly reduces the number of parameters while often maintaining or even improving performance. Our mini GPT also uses this technique.

## Implementation


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/model.py
"""Mini GPT: Embedding + N Transformer Blocks + Output Layer"""
import torch
import torch.nn as nn
from mini_gpt.embedding import GPTEmbedding
from mini_gpt.transformer_block import TransformerBlock


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=4, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.embedding = GPTEmbedding(vocab_size, embed_dim, max_seq_len, dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, max_seq_len, dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight sharing: Token embedding and the output layer use the same matrix
        self.lm_head.weight = self.embedding.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        # Initialization similar to GPT-2: Normal distribution with small standard deviation
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids, targets=None):
        # token_ids: (batch, seq_len)
        x = self.embedding(token_ids)          # (batch, seq_len, embed_dim)

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)               # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            # logits: (batch, seq_len, vocab_size) -> (batch*seq_len, vocab_size)
            # targets: (batch, seq_len) -> (batch*seq_len,)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


## Model Creation and Parameter Count Check


In [ ]:
# ── Colab Cell ──
from mini_gpt.model import MiniGPT
import torch

model = MiniGPT(
    vocab_size=500,     # Vocabulary size of the character tokenizer
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=64,
    dropout=0.1,
)

print(model)
print("\nTotal parameters:", f"{model.num_parameters():,}")


## Verifying the Forward Pass


In [ ]:
# ── Colab Cell ──
# Check if forward pass works correctly with a dummy batch
batch_size = 4
seq_len = 16

fake_input = torch.randint(0, 500, (batch_size, seq_len))
fake_target = torch.randint(0, 500, (batch_size, seq_len))

logits, loss = model(fake_input, targets=fake_target)

print("Input shape:", fake_input.shape)
print("Logits shape:", logits.shape)        # (4, 16, 500)
print("Loss value:", loss.item())
print("\nInterpreting Loss: Since it is a random initialization state before training, -log(1/500) =",
      round(torch.log(torch.tensor(500.0)).item(), 3), "is the expected value.")


If the loss value is near `log(vocab_size)`, it is a signal that the model has been initialized correctly. This is because, before training, the model predicts an equal probability ($1/\text{vocab\_size}$) for all tokens. If this value is much larger or results in `nan`, it indicates a bug in initialization or implementation, so it is crucial to verify this here before moving on.

## Experimenting by Changing Model Scale


In [ ]:
# ── Colab Cell ──
# Compare how the number of parameters changes by varying hyperparameters
configs = [
    {"embed_dim": 64,  "num_heads": 2, "num_layers": 2},
    {"embed_dim": 128, "num_heads": 4, "num_layers": 4},
    {"embed_dim": 256, "num_heads": 8, "num_layers": 6},
]

for cfg in configs:
    m = MiniGPT(vocab_size=500, max_seq_len=64, **cfg)
    print(cfg, "-> Parameter count:", f"{m.num_parameters():,}")


Observe how the number of parameters increases as you increase `embed_dim` or `num_layers`. Understanding this relationship will give you an intuition for why real-world models like GPT-3 (175 billion) and GPT-4 have so many parameters (due to stacking very deep layers and very wide dimensions).

In the next chapter, we will implement the training loop to train this model on actual text data.


# Implementing the Training Loop

Now, let's train our mini-GPT on actual text. The core idea of a training loop is simple: **feed a chunk of text into the model, task it with predicting "the very next token" at each position, and slightly adjust the weights based on how wrong it was**, repeating this process hundreds or thousands of times.

## Preparing Training Data: Shifting Inputs and Targets by One Position

In language model training, there is no need for humans to create separate target labels. The original text itself serves as the ground truth. For example, if a token sequence is `[I, am, eating, food]`:

- Input: `[I, am, eat, food]`
- Target: `[am, eat, food, <EOS>]` (The input shifted one position to the left)

In other words, at the $i$-th position of the input, the model must predict the $i$-th token of the target (=the $(i+1)$-th token of the input). This ability to generate massive amounts of training data using only the original text without manual labeling is a major advantage of language model pre-training (this is called **self-supervised learning**).

## Implementing the Dataset Class


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/dataset.py
"""Dataset that creates training batches of (input, target) from text"""
import torch
from torch.utils.data import Dataset


class TextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        """token_ids: A long list of integers representing the tokenized full text"""
        self.token_ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        # Number of starting positions to extract non-overlapping chunks of size seq_len+1
        return len(self.token_ids) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.token_ids[idx: idx + self.seq_len + 1]
        x = chunk[:-1]   # Input: the first seq_len tokens
        y = chunk[1:]    # Target: the next seq_len tokens (shifted by one)
        return x, y


## Preparing Training Data (Running in Colab)

This time, we will use parts of Korean Wikipedia documents as example data. Since this is for practice, we are using short public texts, but you can replace them with any text files you desire for your own projects.


In [ ]:
# ── Colab Cell ──
from datasets import load_dataset

# Using a tiny English subset of WikiText for practice (fast download)
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:2%]")
text_corpus = "\n".join([t for t in raw_dataset["text"] if len(t.strip()) > 0])
print("Total character count:", len(text_corpus))
print("Sample:\n", text_corpus[:500])


In [ ]:
# ── Colab Cell ──
# Train the BPE tokenizer created in Chapter 2 on this corpus
from mini_gpt.tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tokenizer.train(text_corpus, vocab_size=2000)

all_token_ids = tokenizer.encode(text_corpus)
print("Total token count:", len(all_token_ids))
print("Vocabulary size:", len(tokenizer.vocab))


In [ ]:
# ── Colab Cell ──
from mini_gpt.dataset import TextDataset
from torch.utils.data import DataLoader

seq_len = 64
dataset = TextDataset(all_token_ids, seq_len=seq_len)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(dataloader))
print("Input batch shape:", x_batch.shape)   # (32, 64)
print("Target batch shape:", y_batch.shape)   # (32, 64)


## The Training Loop

Each step of training consists of the following five stages:

1. **Forward Pass**: Feed inputs into the model to calculate predictions (logits) and loss.
2. **Backward Pass (Backpropagation)**: Calculate how much each parameter contributed to the loss (gradients).
3. **Gradient Reset**: Initialize gradients to zero so that they do not accumulate from previous steps.
4. **Optimizer Step**: Move parameters slightly in the direction of the calculated gradients.
5. Repeat with the next batch.


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/train.py
"""Training Loop"""
import torch


def train_model(model, dataloader, num_epochs, learning_rate, device):
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    # Gradually reduce the learning rate towards the end of training for more stable convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(dataloader)
    )

    history = []
    step = 0
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            logits, loss = model(x_batch, targets=y_batch)

            optimizer.zero_grad()
            loss.backward()
            # Prevent gradient explosion: limit the gradient magnitude to a specific range
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step += 1
            history.append(loss.item())

            if step % 50 == 0:
                print(f"epoch {epoch+1} | step {step} | loss {loss.item():.4f}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"=== epoch {epoch+1} average loss: {avg_loss:.4f} ===")

    return history


## Let's Actually Train It


In [ ]:
# ── Colab Cell ──
from mini_gpt.model import MiniGPT
from mini_gpt.train import train_model
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=seq_len,
    dropout=0.1,
)
print("Number of parameters:", f"{model.num_parameters():,}")

history = train_model(
    model,
    dataloader,
    num_epochs=3,
    learning_rate=3e-4,
    device=device,
)


## Checking the Learning Curve


In [ ]:
# ── Colab Cell ──
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Trend")
plt.grid(True, alpha=0.3)
plt.show()


If the loss steadily trends downward, it means the model is learning patterns in the data (frequently occurring word combinations, grammatical structures). It is normal for the initial loss to start near `log(vocab_size)` and decrease as training progresses.

## Saving Checkpoints


In [ ]:
# ── Colab Cell ──
import pickle

torch.save(model.state_dict(), "mini_gpt_checkpoint.pt")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer have been saved.")


Since Colab sessions are reset after a certain period, it is recommended to mount Google Drive using `from google.colab import drive; drive.mount('/content/drive')` and save important checkpoints to that path.

In the next chapter, we will cover how to generate actual text with the trained model and the sampling strategies that determine the quality of the generated results.


# Text Generation and Sampling Strategies

## Generation is the Iteration of "Next Token Prediction"

A trained model outputs a probability distribution (logits) for "what the next token will be" given a sequence of tokens. Text generation is the process of repeating this cycle.

1. Feed the current token sequence into the model.
2. Convert the logits at the last position into a probability distribution.
3. Select a single next token from that distribution.
4. Append the selected token to the end of the sequence.
5. Repeat steps 1–4 until the desired length is reached.

Step 3, "how to select the next token from the distribution," significantly determines the quality and creativity of the generated text. Even with the same model, different sampling strategies produce entirely different styles of writing.

## Sampling Strategies

### Greedy Decoding
Always selects the token with the highest probability. This produces the most "safe" sentences but often results in repetitive and boring text.

### Temperature
Divides the logits by a temperature value before applying softmax.

```
probability = softmax(logits / temperature)
```

- `temperature < 1`: The probability distribution becomes sharper $\rightarrow$ confident (but monotonous) output.
- `temperature > 1`: The probability distribution becomes flatter $\rightarrow$ diverse, but with an increased risk of becoming nonsensical.
- `temperature = 1`: Uses the original distribution as learned by the model.

### Top-k Sampling
Only keeps the top $k$ tokens with the highest probabilities as candidates, sets the probability of all others to zero, and then samples within that subset. This prevents nonsensical tokens from being selected, even if they have a low probability.

### Top-p (Nucleus) Sampling
Adds tokens in descending order of probability until the cumulative probability exceeds $p$ (e.g., 0.9), and then samples within that set. While Top-$k$ fixes the number of candidates, Top-$p$ dynamically adjusts—keeping fewer candidates when the distribution is sharp and more when it is flat. This is the most widely used method in real-world services.

## Implementation


In [ ]:
# ── Colab Cell ──
%%writefile mini_gpt/generate.py
"""Text Generation: Greedy, Temperature, Top-k, and Top-p Sampling"""
import torch
import torch.nn.functional as F


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50,
             temperature=1.0, top_k=None, top_p=None, device="cpu"):
    model.eval()
    model.to(device)

    token_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Use only recent tokens to ensure they do not exceed the maximum length seen during model training
        context = token_ids[:, -model.max_seq_len:]

        logits, _ = model(context)
        last_logits = logits[:, -1, :]  # Logits at the last position: (batch, vocab_size)

        # Apply temperature
        last_logits = last_logits / max(temperature, 1e-5)

        # Top-k filtering: set probabilities to -inf for everything except the top k
        if top_k is not None:
            topk_values, _ = torch.topk(last_logits, top_k)
            threshold = topk_values[:, -1].unsqueeze(-1)
            last_logits = torch.where(
                last_logits < threshold,
                torch.full_like(last_logits, float("-inf")),
                last_logits,
            )

        # Top-p (nucleus) filtering
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(last_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # Mark for removal from the point where cumulative probability exceeds top_p
            remove_mask = cumulative_probs > top_p
            # Always keep the first token to prevent cases with no candidates
            remove_mask[:, 1:] = remove_mask[:, :-1].clone()
            remove_mask[:, 0] = False

            sorted_logits[remove_mask] = float("-inf")
            last_logits = torch.full_like(last_logits, float("-inf"))
            last_logits.scatter_(1, sorted_idx, sorted_logits)

        probs = F.softmax(last_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # Stochastic sampling

        token_ids = torch.cat([token_ids, next_token], dim=1)

    generated_ids = token_ids[0].tolist()
    return tokenizer.decode(generated_ids)


## Real Generation Practice


In [ ]:
# ── Colab Cell ──
from mini_gpt.generate import generate

prompt = "The history of"

print("=== Greedy (approaching temperature=0) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.01, device=device))

print("\n=== Temperature 1.0 ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, device=device))

print("\n=== Top-k Sampling (k=20) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, top_k=20, device=device))

print("\n=== Top-p Sampling (p=0.9) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.8, top_p=0.9, device=device))


Since the mini GPT in this book has very small training data and parameter counts, it is difficult to expect grammatically perfect or factually accurate sentences. However, if training has proceeded well, you can observe it plausibly mimicking English word forms, common combinations, and punctuation patterns. The purpose of this practice is not "building a production-grade model," but rather **experiencing how these same principles scale up to become models like ChatGPT when applied at a much larger scale.**

## Comparative Observation by Parameter


In [ ]:
# ── Colab Cell ──
# Compare how results differ with the same prompt but different temperatures
for temp in [0.3, 0.7, 1.2]:
    print(f"\n--- temperature={temp} ---")
    for _ in range(2):
        print(generate(model, tokenizer, prompt, max_new_tokens=30,
                        temperature=temp, top_p=0.9, device=device))


You can observe that as the temperature decreases, the results of two generation runs become more similar, and as it increases, the results become more diverse and unpredictable each time.

In the next chapter, we will use the Hugging Face library to verify that this architecture we built from scratch is fundamentally identical to pre-trained models containing billions of parameters.


# Working with Real Pre-trained Models using Hugging Face

The mini GPT we have built so far has a parameter count in the hundreds of thousands to millions and uses very small training datasets. In this chapter, we will use the Hugging Face `transformers` library to load actual open-source models that are **structurally identical** to what we built, but trained on a much larger scale with significantly more data. By comparing them with our custom architecture, you will realize that "the principles are exactly the same; only the scale differs."

## Basic Usage of the `transformers` Library


In [ ]:
# ── Colab Cell ──
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"  # Small open model with 120 million parameters

hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_model.eval()

print("Number of parameters:", f"{sum(p.numel() for p in hf_model.parameters()):,}")
print(hf_model.config)


If you print `hf_model.config`, you will see values such as `n_embd` (embedding dimension), `n_layer` (number of layers), and `n_head` (number of heads). These correspond exactly to the `embed_dim`, `num_layers`, and `num_heads` we used when building our `MiniGPT`. For GPT-2, these values are 768, 12, and 12, respectively—far larger than our mini GPT.

## Text Generation with Pre-trained Models


In [ ]:
# ── Colab Cell ──
prompt = "The key idea behind the Transformer architecture is"
inputs = hf_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50,
        pad_token_id=hf_tokenizer.eos_token_id,
    )

print(hf_tokenizer.decode(output_ids[0], skip_special_tokens=True))


The `temperature`, `top_p`, and `top_k` arguments in the `generate()` function have the exact same names as those we implemented manually in Chapter 8. The `generate()` function we built was essentially a miniature version of this function.

## Inspecting the Model's Internal Structure


In [ ]:
# ── Colab Cell ──
# Check the list of Transformer blocks inside GPT-2
print(hf_model.transformer.h)  # h = "hidden layers", serves the same role as our self.blocks

print("\nNumber of blocks:", len(hf_model.transformer.h))
print("First block structure:")
print(hf_model.transformer.h[0])


Looking at the output, you can see `attn` (attention), `mlp` (our FeedForward), `ln_1`, and `ln_2` (our ln1, ln2) appearing directly. While variable names and specific implementations may differ slightly, they consist of the exact same components as the `TransformerBlock` we built in Chapter 5.

## Extracting and Visualizing Attention Weights


In [ ]:
# ── Colab Cell ──
import matplotlib.pyplot as plt

inputs = hf_tokenizer("The cat sat on the mat because it was tired", return_tensors="pt")
with torch.no_grad():
    outputs = hf_model(**inputs, output_attentions=True)

# outputs.attentions: Tuple of tensors per layer (batch, num_heads, seq_len, seq_len)
first_layer_attn = outputs.attentions[0][0]  # (num_heads, seq_len, seq_len)
tokens = hf_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(first_layer_attn[i].numpy(), cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"Head {i}")
plt.tight_layout()
plt.show()


You can observe how each head connects words using different patterns. For example, one head might react strongly to a pronoun relationship where "it" refers to "cat." This is the real-world manifestation of what we described in Chapter 4: "Multi-Head Attention captures multiple perspectives simultaneously."

## Summary of Differences: Our Model vs. GPT-2

| Item | This Book's MiniGPT | GPT-2 (small) |
|---|---|---|
| Embedding Dimension | 64~128 | 768 |
| Number of Layers | 4~6 | 12 |
| Number of Heads | 4 | 12 |
| Parameter Count | Hundreds of thousands to millions | 124 million |
| Training Tokens | Tens of thousands | ~10 billion |
| Vocabulary Size | 500~2,000 | 50,257 |

As shown in the table, the "type" of architecture is identical; only the "scale" differs. This is the core of LLM scaling: an empirical law (Scaling Law) repeatedly confirmed in recent years of LLM research—that performance improves in a predictable manner as you increase data, parameters, and compute while maintaining the same architecture.

In the next chapter, instead of training these pre-trained models from scratch, we will cover fine-tuning to adapt them to specific purposes with minimal resources, specifically focusing on a lightweight technique called **LoRA**.


# Lightweight Fine-tuning with LoRA

## Why Full Fine-tuning is Burdensome

To adapt a pre-trained model to specific tasks (e.g., responding in a specific tone or incorporating domain-specific knowledge), additional training is required. The most intuitive method is to retrain **all parameters of the model**, but this presents several practical challenges:

- As models grow larger, memory consumption becomes massive because you must load the gradients and optimizer states for every single parameter into GPU memory.
- Each fine-tuned result requires storage space equal to the size of the entire original model.
- When data is scarce, updating all parameters can easily lead to **catastrophic forgetting**, where the model loses its previously learned knowledge.

## The Core Idea of LoRA

**LoRA (Low-Rank Adaptation)** keeps the weights of the original model frozen and instead attaches two very small matrices ($A$ and $B$) next to each layer, training only those small matrices.

Original weight updates can be expressed as follows:

```
New Weight = Original Weight W + ΔW
```

In standard fine-tuning, $\Delta W$ is a full matrix of the same size as $W$. LoRA approximates $\Delta W$ as the product of two much smaller matrices.

```
ΔW ≈ B · A     (A: r×d size, B: d×r size, r is much smaller than d, e.g., r=8)
```

The smaller the rank ($r$), the more dramatically the number of parameters to be trained decreases. For example, fine-tuning an entire layer with $d=768$ requires approximately $768 \times 768 \approx 590,000$ parameters, whereas a LoRA with $r=8$ requires only $768 \times 8 + 8 \times 768 \approx 12,000$—training only about 2% of the parameters. Remarkably, this method achieves performance comparable to full fine-tuning in many tasks.

This approach provides several advantages:

- The number of trainable parameters and GPU memory usage are significantly reduced.
- Since the original weights remain unchanged, you only need to save the LoRA adapters (matrices $A$ and $B$), making the output size tiny—around a few dozen MBs.
- You can create multiple LoRA adapters and swap them out depending on the situation.

## Hands-on with Hugging Face's `peft` Library


In [ ]:
# ── Colab Cell ──
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                       # LoRA rank: Smaller values result in fewer parameters
    lora_alpha=16,             # Scaling factor (usually 2x r)
    lora_dropout=0.05,
    target_modules=["c_attn"],  # Apply LoRA only to attention layers that create Q, K, V in GPT-2
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()


Running `print_trainable_parameters()` outputs the ratio of trainable parameters relative to the total parameters. You will typically see that it is less than 1%. This is why LoRA is referred to as "lightweight" fine-tuning.

## Fine-tuning with a Simple Dataset

Here, we use a small amount of example data to train the model to respond in a specific tone (concise and assertive).


In [ ]:
# ── Colab Cell ──
from torch.utils.data import Dataset, DataLoader

training_examples = [
    "Q: What is a transformer?\nA: A transformer is a neural network built entirely on attention.",
    "Q: What is attention?\nA: Attention lets a model weigh how much each token matters to another.",
    "Q: What is a tokenizer?\nA: A tokenizer converts raw text into integer tokens a model can process.",
    "Q: What is fine-tuning?\nA: Fine-tuning adapts a pretrained model to a new task using extra training.",
] * 30  # Repeat small datasets to learn patterns


class SimpleTextDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len=64):
        self.encodings = tokenizer(
            examples, truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt",
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        input_ids = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": input_ids.clone(),
        }


train_dataset = SimpleTextDataset(training_examples, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [ ]:
# ── Colab Cell ──
device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model.to(device)
peft_model.train()

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)

for epoch in range(3):
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = peft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1} average loss: {total_loss/len(train_loader):.4f}")


## Saving and Reusing LoRA Adapters


In [ ]:
# ── Colab Cell ──
peft_model.save_pretrained("gpt2-lora-adapter")
print("Adapter saving complete. Check the folder size (it is much smaller than the original model).")

import os
total_size = sum(
    os.path.getsize(os.path.join("gpt2-lora-adapter", f))
    for f in os.listdir("gpt2-lora-adapter")
)
print(f"Adapter folder size: {total_size / 1024 / 1024:.2f} MB")


In [ ]:
# ── Colab Cell ──
# How to reload the saved adapter later and attach it to the original model
from peft import PeftModel

fresh_base_model = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(fresh_base_model, "gpt2-lora-adapter")

loaded_model.eval()
prompt = "Q: What is a transformer?\nA:"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    output = loaded_model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))


As shown, LoRA allows you to train, save, and swap small adapters while leaving the original model (whether it's GPT-2, LLaMA-family, Qwen, etc.) intact. This makes it a practical method for experiencing real-world fine-tuning even in resource-constrained environments like personal GPUs or Colab free tier.

In the next chapter, we will explore **Prompt Engineering** techniques that guide models to desired responses without changing their weights at all.


# Prompt Engineering and Inference-time Control

Fine-tuning is a method of changing the model's weights themselves. In contrast, **Prompt Engineering** is a technique that guides the model's output in a desired direction solely by designing the input (prompt) without touching the weights at all. Because it is highly effective and costs almost nothing, it is the first approach attempted in practice.

## Zero-shot vs Few-shot

- **Zero-shot**: Providing instructions directly without any examples. `"Translate the following sentence into Korean: Hello"`
- **Few-shot**: Including a few examples of the desired input-output format within the prompt. The model observes the pattern of these examples and follows that format.


In [ ]:
# ── Colab Cell ──
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

def generate_text(prompt, max_new_tokens=30, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


zero_shot_prompt = "Translate to French: Good morning"
print("=== Zero-shot ===")
print(generate_text(zero_shot_prompt))

few_shot_prompt = (
    "English: Hello -> French: Bonjour\n"
    "English: Thank you -> French: Merci\n"
    "English: Good morning -> French:"
)
print("\n=== Few-shot ===")
print(generate_text(few_shot_prompt))


Because few-shot prompting directly shows the model "what task it is performing and what format the answer should take" through examples, it produces much more stable results than zero-shot, especially for smaller models. This phenomenon is possible because the model has already learned extensive pattern repetition within vast amounts of text during its pre-training stage.

## Role Assignment and System Prompts

Conversational models like ChatGPT use **System Prompts**—such as "You are a skilled coding assistant"—at the beginning of a conversation to specify the model's role and response style. The principle is similar to few-shot prompting: by laying out context at the very beginning of the model input, the probability that the subsequent generation follows that context increases.


In [ ]:
# ── Colab Cell ──
system_context = (
    "You are a concise technical writer. You answer in exactly one short sentence.\n\n"
)
question = "Q: What is gradient descent?\nA:"

print(generate_text(system_context + question, max_new_tokens=25))


## Chain-of-Thought (CoT) Prompting

For problems requiring complex reasoning, it has been observed that models tend to produce more accurate answers when prompted to "think step-by-step" rather than being told to "give the answer immediately." This is called Chain-of-Thought (CoT) prompting.


In [ ]:
# ── Colab Cell ──
direct_prompt = "Q: If a train travels 60 miles in 1.5 hours, what is its speed? A:"
cot_prompt = (
    "Q: If a train travels 60 miles in 1.5 hours, what is its speed?\n"
    "A: Let's think step by step. Speed equals distance divided by time. "
    "Distance is 60 miles, time is 1.5 hours. So speed ="
)

print("=== Direct Question ===")
print(generate_text(direct_prompt, max_new_tokens=20))
print("\n=== Chain-of-Thought Prompting ===")
print(generate_text(cot_prompt, max_new_tokens=20))


While this may appear to have limited effectiveness in the Mini GPT or small GPT-2 models used in this book, multiple studies have reported that the effects of CoT prompting become significantly more pronounced as model parameter size increases. This is cited as a representative example of "emergent ability" that appears as model scale grows.

## Summary of Inference Parameters

Here is a summary of the generation parameters learned so far.

| Parameter | Role | Lowering Value | Raising Value |
|---|---|---|---|
| `temperature` | Controls sharpness of probability distribution | More deterministic, repetitive | More diverse, random |
| `top_k` | Limits number of candidate tokens | Fewer candidates, safer selection | Allows more diverse candidates |
| `top_p` | Limits candidates based on cumulative probability | Fewer candidates | Allows a wider range of candidates |
| `max_new_tokens` | Maximum number of tokens to generate | Shorter responses | Longer responses (increased cost/time) |

In practice, it is common practice to set a low `temperature` for tasks with clear correct answers (code generation, fact-based Q&A) and a relatively high `temperature` for creative tasks (storytelling, brainstorming).

Next, in the final chapter, we will summarize this entire book and suggest learning paths for readers who wish to go further.


# Conclusion: Summary and Next Steps

## Looking Back at What We Built

In this book, we implemented each module of the pipeline one by one to understand how a single line of text actually undergoes the process of predicting the next word.

```
mini_gpt/
├── tokenizer.py          # Chapter 2: Text -> Token IDs (BPE)
├── embedding.py           # Chapter 3: Token IDs -> Vectors (Token Embedding + Positional Embedding)
├── attention.py            # Chapter 4: Self-Attention, Multi-Head Attention
├── transformer_block.py    # Chapter 5: Attention + FeedForward + LayerNorm + Residual
├── model.py                 # Chapter 6: Full GPT Model composed of N blocks
├── dataset.py                # Chapter 7: Converting text into (input, target) pairs
├── train.py                   # Chapter 7: Training loop
└── generate.py                 # Chapter 8: Text generation and sampling strategies
```

In Chapters 9 through 11, we confirmed that these principles apply identically to pre-trained models like GPT-2, and saw how we can lightly fine-tune them with LoRA or control model behavior using only prompts without touching the weights.

## Summary of Core Concepts

- **Tokenization**: Splitting text into integer units that a model can process. BPE (Byte Pair Encoding) builds a vocabulary by merging frequently occurring fragments into larger units.
- **Embedding**: Converting token IDs into semantic vectors and adding positional embeddings so that order information is preserved.
- **Attention**: Using a Query-Key-Value structure to calculate how much each token should attend to other tokens in the context. A Causal Mask hides future tokens to establish the "next-word prediction" task.
- **Transformer Block**: Stabilizing training with residual connections and LayerNorm, and increasing expressivity by repeating Attention (token relationships) and FeedForward (per-token transformations).
- **Training**: A self-supervised learning method where the original text is shifted by one position to serve as the ground truth, allowing for the use of massive datasets without manual human labeling.
- **Generation**: The character of the output changes depending on how next tokens are selected from the learned probability distribution using temperature, top-k, and top-p.
- **Fine-tuning and LoRA**: Adjusting a model to specific purposes with minimal resources by training only small adapter matrices rather than the entire model.
- **Prompt Engineering**: Guiding model output through input design without changing the weights.

## Differences from Real Large Language Models (LLMs) and Topics for Further Study

The Mini GPT in this book is an intentionally scaled-down model designed for learning principles. To move toward production-grade LLMs, we recommend exploring the following topics:

- **Efficient Attention**: Techniques like Flash Attention and Grouped-Query Attention (GQA) that process long contexts faster and more memory-efficiently.
- **Advances in Positional Encoding**: Positional encoding methods like RoPE (Rotary Position Embedding) that generalize better to longer context lengths.
- **RLHF and Alignment**: The process of using Reinforcement Learning from Human Feedback to tune models to provide useful and safe responses.
- **Quantization**: Techniques to reduce memory and computation by representing model weights with fewer bits (e.g., INT8, INT4).
- **MoE (Mixture of Experts)**: An architecture that activates only a fraction of the total parameters for each input, allowing for larger model capacity relative to computational cost.
- **RAG (Retrieval-Augmented Generation)**: A technique that combines real-time information into model responses by searching an external knowledge base.
- **Evaluation**: Methodologies for quantitatively measuring a model's reasoning, knowledge, and safety using benchmark datasets.

## Recommended Learning Path

1. In this book's `MiniGPT`, try increasing `embed_dim`, `num_layers`, and `num_heads` to observe the relationship between parameter count, training time, and generation quality.
2. Re-train a tokenizer and model using your own Korean corpus (news, blogs, novels, etc.).
3. Compare full fine-tuning with LoRA fine-tuning results using Hugging Face's `Trainer` API.
4. Read the original "Attention Is All You Need" and GPT-2 ("Language Models are Unsupervised Multitask Learners") papers, and map the equations in the papers to the code implemented in this book one by one.

## Closing Remarks

LLMs may seem like magic, but as we have seen in this book, they consist of well-defined components such as tokenization, matrix multiplication, softmax, and gradient descent. While incredible capabilities emerge as scale increases, the foundation is exactly the structure you have manipulated through code in this book. Building upon this foundation and reading more advanced papers and open-source code will make it much easier to understand future advancements.


# Glossary

This section provides definitions for the key terms related to artificial intelligence and Large Language Model (LLM) technologies covered in this book.

* **BPE (Byte Pair Encoding)**: A subword-based tokenizer algorithm that builds a token vocabulary by iteratively merging the most frequently occurring byte pairs in the data. It solves the OOV (Out-Of-Vocabulary) problem.
* **Embedding**: A technique that maps natural language words or tokens into high-dimensional real-valued vector spaces that computers can understand. It reflects semantic similarity between words.
* **Attention**: A mechanism that dynamically determines which tokens in an input sequence the model should focus on (assign weights to).
* **Self-Attention**: A method where tokens within the same sequence calculate weights relative to one another, quantifying their interrelationships and meanings within a context.
* **Multi-Head Attention**: An architecture that splits attention into multiple "heads" to process information in parallel across different representation subspaces before combining them.
* **LayerNorm (Layer Normalization)**: A technique that normalizes activation values based on the mean and variance of features within each data sample. It stabilizes training and accelerates convergence.
* **Residual Connection**: A skip connection structure that adds the input of a layer directly to its output. This significantly mitigates the vanishing gradient problem, enabling the training of deep networks.
* **Transformer Block**: The fundamental building block of the Transformer architecture, consisting of Multi-Head Attention, Layer Normalization, Residual Connections, and a Feed-Forward Network (FFN).
* **Decoder**: A component of the Transformer that includes a masked Self-Attention structure to sequentially predict the next token (auto-regressive) based on previous sequence information. It forms the backbone of GPT-style models.
* **Sampling Strategy**: Methods used to control randomness and diversity when a generative model selects the next token.
  - **Temperature**: A hyperparameter that adjusts the softmax distribution to govern the creativity and determinism of the generated text.
  - **Top-K**: A technique that restricts sampling to only the top $K$ tokens with the highest probabilities.
  - **Top-P (Nucleus Sampling)**: A technique that selects a minimum set of candidates whose cumulative probability reaches threshold $P$, then samples from within that set.
* **LoRA (Low-Rank Adaptation)**: An efficient Parameter-Efficient Fine-Tuning (PEFT) technique that decomposes weight update matrices into two smaller low-rank matrices, reducing the number of trainable parameters by over 99%.
